# Fetch and plot finetune results

### imports, functions etc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re, os, pickle
import pandas as pd
import seaborn as sns

: 

In [ ]:
def plot_finetuning_results(folder_path, xlabel, ylabel, title_notes, title, simple_model_results, xgb_results, lo, hi, custom_order):
    """
    Loads finetuning results from a given folder, applies word substitutions,
    and creates a line plot of Macro F1 scores vs. training size.

    Parameters:
        folder_path (str): Path to the results folder.
        xlabel (str): Label for the x-axis.
        ylabel (str): Label for the y-axis.
        title_notes (str): Additional notes to be appended to the plot title.
        title (str): Main title for the plot.
        simple_model_results (list): list of dicts with keys: "n_comps" "model_type" "result_dict"
        include_xgb (bool): whether or not to include xgb
        custom_order (list): Define custom order for legend display and add the new category
        lo (str): low bound for cell numbers, used to fetch the corresponding xgb results
        hi (str): high bound for cell numbers, used to fetch the corresponding xgb results

    Returns:
        df (pandas.DataFrame): The processed DataFrame used for plotting.
    """
    # Load the finetuning results from the specified folder
    df = finetuning_res_to_df_raw(folder_path, "macro_f1")
    
    # Add simple model results
    if simple_model_results:
        df2 = parse_simple_model_results(simple_model_results)
        df = pd.concat([df, df2],axis=0, ignore_index=True)
        
    if xgb_results:
        df3 = scan_xgb_results(xgb_results, lo, hi, "macro_f1")
        df = pd.concat([df, df3],axis=0, ignore_index=True)

    # Define the replacement dictionary for renaming pretraining methods
    replacements = {
        "RNAquarium": "Geneformer: RNAquarium 60k bulk transcriptomes",
        "randomized": "Geneformer: random guassian data",
        "scrambled": "Geneformer: randomly shuffled RNAquarium data",
        "human": "Geneformer: human 30M scRNAseq(Theodoris et al.)",
        "initialweights": "Geneformer: no pretraining, use intial weights",
        #"xgboost_training": "XGBboost: using Geneformer preprocessing",
        #"simple model": "Nearest neighbor (simple model)"
    }

    # Apply word substitutions in the 'pretraining' column using the replacement dictionary
    df = substitute_words(df, "pretraining", replacements)

    # Ensure 'TrainSize' is integer type for plotting purposes
    df["TrainSize"] = df["TrainSize"].astype(int)
    
    # Define custom order for legend display and add the new category
    if not custom_order:
        custom_order = [
            "Geneformer: RNAquarium 60k bulk transcriptomes", 
            "Geneformer: no pretraining, use intial weights",
            "Geneformer: human 30M scRNAseq(Theodoris et al.)",
            "Geneformer: randomly shuffled RNAquarium data",
            "Geneformer: random guassian data",
        ]

    # Create the line plot and add dashes mapping for the new category
    plt.figure(figsize=(8, 10))
    sns.lineplot(
        data=df,
        x="TrainSize",
        y="metric_value",
        hue="pretraining",
        hue_order=custom_order,
        #dashes={"xgboost_training": (2, 2)},  # Set dashed line for xgboost_training
        err_style="bars",
        err_kws={"capsize": 3},
        errorbar=("sd")  # Standard deviation as error bars
    )

    # Set the provided labels and title
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"{title}\n{title_notes}")

    plt.legend(title="Model and pretraining data")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.show()

    return df

In [ ]:
def finetuning_res_to_df_raw(dir_to_scan, metric):
    """
    Scan the folder and get raw metric data into a dataframe
    """
    data = scan_folder(dir_to_scan, metric)[0]

    parsed_data = []

    for key, value_dict in data.items():
        parts = key.split('_')
        
        if len(parts) < 5:
            print(f"skipping {key}")
            continue
                
        if len(parts) == 5:
            second_item = parts[1]
            fourth_item = "RNAquarium"
            sixth_item = parts[4]
            input_size = ""
            epoch_num = ""

        elif len(parts) == 6:
            second_item = parts[1]
            fourth_item = parts[3]
            sixth_item = parts[5]
            input_size = ""
            epoch_num = ""
            
        elif len(parts) == 8: # pretraining strategy scan
            second_item = parts[1]
            fourth_item = "RNAquarium"
            sixth_item = parts[4]
            input_size = parts[6]
            epoch_num = parts[7]
        
        sixth_item = re.findall(r'\d+', sixth_item)

        sixth_item = 100 - int(sixth_item[0])  # Convert to integer
        values = list(value_dict.values())
        for value in values:
            parsed_data.append((second_item, fourth_item, sixth_item, input_size, epoch_num, value))
            #print(f"{second_item}, {fourth_item}, {sixth_item}")

    # Convert to DataFrame for easier plotting
    df = pd.DataFrame(parsed_data, columns=['finetuning_task', 'pretraining', 'TrainSize', "inputSize", "epoch_num", 'metric_value'])
       
    return df

In [ ]:
def scan_folder(base_path, dict_key, secondary_dir_skip_list = []):
    """
    Scan up to two levels of subfolders, using the first level folder name as the observation name.
    The second level folder names are used as repeated observations.
    Reads files ending with "_test_metrics_dict.pkl" into a dictionary.
    Also returns a list of (first-level, second-level) folder names where no such file exists.
    Additionally, checks for "model.safetensors" in a "ksplit" folder at the fourth level.

    :param base_path: Path to the main directory to scan
    :param dict_key: target key to extract
    :return: A tuple containing a dictionary with collected test metrics, a list of missing file folders, and a list of missing ksplit model folders
    """
    results = {}
    missing_files = []
    missing_ksplit_models = []

    for obs_name in os.listdir(base_path):
        obs_path = os.path.join(base_path, obs_name)
        if not os.path.isdir(obs_path):
            continue  # Skip files, only process directories
        
        results[obs_name] = {}
        
        for rep_obs_name in os.listdir(obs_path):
            rep_obs_path = os.path.join(obs_path, rep_obs_name)
            
            if rep_obs_name in secondary_dir_skip_list:
                continue
            
            if not os.path.isdir(rep_obs_path):
                continue  # Skip files, only process directories
            
            # Look for a file ending with "_test_metrics_dict.pkl" but not with "xgboost_test_metrics_dict.pkl"
            pkl_file = next(
                (f for f in os.listdir(rep_obs_path)
                 if f.endswith("_test_metrics_dict.pkl") and not f.endswith("xgboost_test_metrics_dict.pkl")),
                None
            )
            if pkl_file:
                pkl_path = os.path.join(rep_obs_path, pkl_file)
                try:
                    with open(pkl_path, 'rb') as f:
                        res_dict = pickle.load(f)
                        if dict_key in res_dict:
                            results[obs_name][rep_obs_name] = res_dict[dict_key]
                except Exception as e:
                    print(f"Error loading {pkl_path}: {e}")
            else:
                missing_files.append((obs_name, rep_obs_name))
            
            # Enumerate third-level folders to find "ksplit" and check for "model.safetensors"
            model_exist_flag = 0
            for third_level_name in os.listdir(rep_obs_path):
                third_level_path = os.path.join(rep_obs_path, third_level_name)
                if not os.path.isdir(third_level_path):
                    continue  # Skip files, only process directories
                
                ksplit_path = os.path.join(third_level_path, "ksplit1")
                #print(f"checking path {ksplit_path}")
                if os.path.isdir(ksplit_path):
                    if "model.safetensors" in os.listdir(ksplit_path):
                        model_exist_flag = 1
            if model_exist_flag == 0:
               missing_ksplit_models.append((obs_name, rep_obs_name, third_level_name))
    
    return results, missing_files, missing_ksplit_models


In [ ]:
def parse_simple_model_results(simple_model_results):
    """
    simple_model_results (dictionary): contain a key named "all_f1_scores" 
    """
    dfs = []
    for item in simple_model_results: # loop through each configuration (e.g. nComps)
        result_dict = item["result_dict"]
        model_type = item["model_type"]
        n_comps = item["n_comps"]
        data = []
        for replicate in result_dict['all_f1_scores']:
            for frac, score in replicate.items():
                data.append({
                    'finetuning_task': model_type + f" nComps={n_comps}",
                    'pretraining': model_type + f" nComps={n_comps}",
                    'TrainSize': 100 * (float(frac)),  # 1 - key
                    'inputSize': None,            # left empty
                    'epoch_num': None,            # left empty
                    'metric_value': float(score)  # dictionary value
                })
        df = pd.DataFrame(data, columns=[
            'finetuning_task', 'pretraining', 'TrainSize', 'inputSize', 'epoch_num', 'metric_value'
        ])
        dfs.append(df)
    final_df = pd.concat(dfs,axis=0, ignore_index=True)
    return final_df

In [ ]:
def substitute_words(df, column, replacements):
    """
    Replace words in a specific column of a pandas DataFrame.

    Parameters:
    df (pd.DataFrame): The dataframe containing the column to modify.
    column (str): The column in which to replace words.
    replacements (dict): A dictionary where keys are words to replace, and values are the new words.

    Returns:
    pd.DataFrame: A new dataframe with the modified column.
    """
    df[column] = df[column].replace(replacements, regex=True)
    return df

In [ ]:
def scan_xgb_results(results_folder, lo, hi, dict_key):
    """
    parse xgb results and prepare a dataframe
    results_folder (string): path to xgb output
    dict_key (string): e.g. "macro_f1"
    """
    print("parsing xgb results")
    data = []
    for content_item in os.listdir(results_folder):
        content_item_path = os.path.join(results_folder, content_item)
        if os.path.isdir(content_item_path):
            #print(f"parsing {content_item_path}")
            # parse subfolder name
            ncomps, testsize, rep = content_item.split("_")
            ncomps = ncomps.replace("ncomps","")
            testsize = testsize.replace("testsize","")
            rep = rep.replace("rep","")
            #print(f"{ncomps}, {testsize}, {rep}")
            # parse results file
            pkl_path = os.path.join(content_item_path, "xgb_test_metrics_dict.pkl")
            if os.path.isfile(pkl_path):
                try:
                    with open(pkl_path, 'rb') as f:
                        res_dict = pickle.load(f)
                        metric_value = res_dict.get(dict_key, "")
                        data.append({
                            'finetuning_task': "XGBoost" + f" nComps={ncomps}",
                            'pretraining': "XGBoost" + f" nComps={ncomps}",
                            'TrainSize': 100 - 100 * (float(testsize)),  # 1 - key
                            'inputSize': None,            # left empty
                            'epoch_num': None,            # left empty
                            'metric_value': float(metric_value)  # dictionary value
                        })

                except Exception as e:
                    print(f"Error loading {pkl_path}: {e}")
    df = pd.DataFrame(data, columns=[
        'finetuning_task', 'pretraining', 'TrainSize', 'inputSize', 'epoch_num', 'metric_value'
    ])
    return df if df.shape[0] > 0 else None

## Zebrahub dev_x_tissue

### XGBoost

In [ ]:
# plot 
lo_bound = 100
hi_bound = 3000
meta_info = "[80/98 classes, 85/120k cells]"
folder_path = f"/path/to/Geneformer_RQfork/finetune/input2048_30epochs_zebrahub_{lo_bound}-{hi_bound}_hyperopted/scan_devtissue"
xgb_res_path = "/path/to/Geneformer_RQfork/simple_model/xgb/results"
xlabel = "% zebrahub data used for finetuning\n(remaining is used in computing Macro F1 score)"
ylabel = "Macro F1 score\n(error bars = standard deviation from 6 repeated finetuning runs)"
title_notes = f"notes: {lo_bound} cells/class min., large classes downsized to {hi_bound}. {meta_info}"
title = "Performance of zebrahub task using deep models pretrained with different data\nzebrahub task: predicting development_stage + tissue"

custom_order = [
    "Geneformer: RNAquarium 60k bulk transcriptomes", 
    "nearest neighbor nComps=20",
    "nearest neighbor nComps=50",
    "nearest neighbor nComps=100",
    "XGBoost nComps=20",
    "XGBoost nComps=50",
    "XGBoost nComps=100",
    
]
    
# Call the function with the example parameters
df_result = plot_finetuning_results(folder_path, xlabel, ylabel, title_notes, title, None, xgb_res_path, lo_bound, hi_bound, custom_order)

### All

In [ ]:
# plot 
lo_bound = 100
hi_bound = 3000
meta_info = "[80/98 classes, 85/120k cells]"

folder_path = f"/path/to/Geneformer_RQfork/finetune/input2048_30epochs_zebrahub_{lo_bound}-{hi_bound}_hyperopted/scan_devtissue"
xgb_res_path = "/path/to/Geneformer_RQfork/simple_model/xgb/results"
xlabel = "% zebrahub data used for finetuning"
ylabel = "Macro F1 score of predicting single-cell labels\nin Zebrahub dataset (development & tissue)"
title_notes = f"notes: {lo_bound} cells/class min., large classes downsized to {hi_bound}. {meta_info}"
title = "Performance of zebrahub task using deep models pretrained with different data\nzebrahub task: predicting development_stage + tissue"

# save path
save_dir = os.path.join(folder_path, "plot_output")
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "f1_plot.svg")

custom_order = [
        "Deep model pretrained with\nRNAquarium expression data",
        "XGBoost model\n(no pretraining)",
        "Deep model pretrained with\nZebrahub data",
        "Deep model pretrained with\nrandom guassian data",
        #"Deep model pretrained with\nshuffled data",
        "Deep model pretrained with\n Human scRNAseq data",
        
]

dashed_map = {
        #"Deep model trained with\nRNAquarium expression data": (4,2),
        "Deep model pretrained with\nZebrahub data": (3,2),
        "Deep model pretrained with\n Human scRNAseq data": (3,2),
        #"XGBoost model trained with\nRNAquarium expression data": (4,2),
        "Deep model pretrained with\nrandom guassian data": (3,2),
    # anything not listed stays solid
}

def plot_finetuning_results_slides(
    folder_path, xlabel, ylabel, title_notes, title,
    simple_model_results, xgb_results, lo, hi, custom_order,
    dashed=dashed_map,                 # (as added earlier)
    dashed_pattern=(4, 2),

    # NEW: legend controls
    legend_loc="best",           # e.g. "upper right", "center left", "best"
    legend_bbox=None,            # e.g. (1.02, 0.5) to place outside at right
    legend_ncol=1,               # columns in the legend
    legend_fontsize=15,

    # NEW: export controls
    export_path=save_path,            # e.g. "figs/zebrahub_performance.svg"
    export_format="svg",         # "svg" | "pdf" | "eps"
    export_dpi=300,              # used if anything rasterizes (safe default)
    export_transparent=True,     # keep background transparent for Illustrator
    keep_text_as_text=True       # for SVG: keep text selectable/editable
):
    """
    Creates the plot and (optionally) saves a vector file for Illustrator.
    """
    # (load/merge/rename data as in your current function) --------------------
    df = finetuning_res_to_df_raw(folder_path, "macro_f1")

    if simple_model_results:
        df2 = parse_simple_model_results(simple_model_results)
        df = pd.concat([df, df2], axis=0, ignore_index=True)

    if xgb_results:
        df3 = scan_xgb_results(xgb_results, lo, hi, "macro_f1")
        df = pd.concat([df, df3], axis=0, ignore_index=True)

    replacements = {
        "RNAquarium": "Deep model pretrained with\nRNAquarium expression data",
        "randomized": "Deep model pretrained with\nrandom guassian data",
        "scrambled": "Deep model pretrained with\nshuffled data",
        "human": "Deep model pretrained with\n Human scRNAseq data",
        "initialweights": "Deep model: no pretraining, use intial weights",
        "XGBoost nComps=20": "XGBoost model\n(no pretraining)",
        "zebrahubData": "Deep model pretrained with\nZebrahub data",
    }

    #print(df["pretraining"].value_counts())
    df = substitute_words(df, "pretraining", replacements)
    df["TrainSize"] = df["TrainSize"].astype(int)

    if not custom_order:
        custom_order = [
            "Deep model w/ RNAquarium expression data",
            "XGBoost model",
            "Deep model w/ random guassian data",
            "Deep model w/ shuffled data",
        ]
    present_levels = [c for c in custom_order if c in set(df["pretraining"])]

    # dash mapping ------------------------------------------------------------
    dash_map = None
    if dashed is not None:
        if isinstance(dashed, dict):
            dash_map = {lvl: dashed.get(lvl, "") for lvl in present_levels}
        else:
            if isinstance(dashed, str):
                dashed = [dashed]
            dashed = set(dashed)
            dash_map = {lvl: (dashed_pattern if lvl in dashed else "") for lvl in present_levels}

    # keep text editable in SVG (nice for Illustrator)
    if export_format == "svg" and keep_text_as_text:
        import matplotlib as mpl
        mpl.rcParams["svg.fonttype"] = "none"

    # plot --------------------------------------------------------------------
    plt.figure(figsize=(8, 8))
    sns.lineplot(
        data=df,
        x="TrainSize",
        y="metric_value",
        hue="pretraining",
        hue_order=present_levels,
        style="pretraining" if dash_map is not None else None,
        style_order=present_levels if dash_map is not None else None,
        dashes=dash_map if dash_map is not None else False,
        err_style="bars",
        err_kws={"capsize": 3},
        errorbar=("sd"),
    )

    # labels & axes
    plt.xlabel(xlabel, fontsize=20)
    plt.ylabel(ylabel, fontsize=20)
    plt.title("")
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)

    # LEGEND: fully adjustable
    if legend_bbox is not None:
        plt.legend(title="", fontsize=legend_fontsize, ncol=legend_ncol,
                   loc=legend_loc, bbox_to_anchor=legend_bbox, borderaxespad=0.0)
    else:
        plt.legend(title="", fontsize=legend_fontsize, ncol=legend_ncol, loc=legend_loc)

    plt.ylim(0, 1)
    plt.grid(False)
    plt.tight_layout()

    # EXPORT: vector formats for Illustrator
    if export_path is not None:
        plt.savefig(
            export_path,
            format=export_format,
            dpi=export_dpi,          # harmless for vector; used if any rasterized parts exist
            bbox_inches="tight",
            transparent=export_transparent,
        )

    plt.show()
    return df


# Call the function with the example parameters
df_result = plot_finetuning_results_slides(folder_path, xlabel, ylabel, title_notes, title, None, xgb_res_path, lo_bound, hi_bound, custom_order)

In [ ]:
# --- NEW: manual color palette (edit these as you like) ----------------------
color_map = {
    "Deep model pretrained with\nRNAquarium expression data": "#df6766", # red
    "XGBoost model\n(no pretraining)":                        "#45bebe",  # teal          #"#f39d49",  # orange
    "Deep model pretrained with\nZebrahub data":                         "#9ba639",  # olive
    "Deep model pretrained with\nrandom guassian data":                  "#707c92",  # gray
    "Deep model pretrained with\n Human scRNAseq data":                  "#936b57",  # skin
}

# (your existing vars)
lo_bound = 100
hi_bound = 3000
meta_info = "[80/98 classes, 85/120k cells]"

folder_path = f"/path/to/Geneformer_RQfork/finetune/input2048_30epochs_zebrahub_{lo_bound}-{hi_bound}_hyperopted/scan_devtissue"
xgb_res_path = "/path/to/Geneformer_RQfork/simple_model/xgb/results"
xlabel = "% zebrahub data used for finetuning"
ylabel = "Macro F1 score of predicting single-cell labels\nin Zebrahub dataset (development & tissue)"
title_notes = f"notes: {lo_bound} cells/class min., large classes downsized to {hi_bound}. {meta_info}"
title = "Performance of zebrahub task using deep models pretrained with different data\nzebrahub task: predicting development_stage + tissue"

# save path
save_dir = os.path.join(folder_path, "plot_output")
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "f1_plot.svg")

custom_order = [
    "Deep model pretrained with\nRNAquarium expression data",
    "XGBoost model\n(no pretraining)",
    "Deep model pretrained with\nZebrahub data",
    "Deep model pretrained with\nrandom guassian data",
    # "Deep model pretrained with\nshuffled data",
    "Deep model pretrained with\n Human scRNAseq data",
]

dashed_map = {
    "XGBoost model\n(no pretraining)": (3, 2),
    "Deep model pretrained with\nZebrahub data": (3, 2),
    "Deep model pretrained with\n Human scRNAseq data": (3, 2),
    "Deep model pretrained with\nrandom guassian data": (3, 2),
}

def plot_finetuning_results_slides(
    folder_path, xlabel, ylabel, title_notes, title,
    simple_model_results, xgb_results, lo, hi, custom_order,
    dashed=dashed_map,
    dashed_pattern=(4, 2),

    # NEW: legend controls
    legend_loc="best",
    legend_bbox=None,
    legend_ncol=1,
    legend_fontsize=15,

    # NEW: export controls
    export_path=save_path,
    export_format="svg",
    export_dpi=300,
    export_transparent=True,
    keep_text_as_text=True,

    # NEW: palette controls
    color_map=color_map,            # dict: {level -> color}
    fallback_palette="tab10"        # used if a level missing in color_map
):
    """
    Creates the plot and (optionally) saves a vector file for Illustrator.
    """
    # (load/merge/rename data as in your current function) --------------------
    df = finetuning_res_to_df_raw(folder_path, "macro_f1")

    if simple_model_results:
        df2 = parse_simple_model_results(simple_model_results)
        df = pd.concat([df, df2], axis=0, ignore_index=True)

    if xgb_results:
        df3 = scan_xgb_results(xgb_results, lo, hi, "macro_f1")
        df = pd.concat([df, df3], axis=0, ignore_index=True)

    replacements = {
        "RNAquarium": "Deep model pretrained with\nRNAquarium expression data",
        "randomized": "Deep model pretrained with\nrandom guassian data",
        "scrambled": "Deep model pretrained with\nshuffled data",
        "human": "Deep model pretrained with\n Human scRNAseq data",
        "initialweights": "Deep model: no pretraining, use intial weights",
        "XGBoost nComps=20": "XGBoost model\n(no pretraining)",
        "zebrahubData": "Deep model pretrained with\nZebrahub data",
    }

    df = substitute_words(df, "pretraining", replacements)
    df["TrainSize"] = df["TrainSize"].astype(int)

    if not custom_order:
        custom_order = [
            "Deep model w/ RNAquarium expression data",
            "XGBoost model",
            "Deep model w/ random guassian data",
            "Deep model w/ shuffled data",
        ]
    present_levels = [c for c in custom_order if c in set(df["pretraining"])]

    # dash mapping ------------------------------------------------------------
    dash_map = None
    if dashed is not None:
        if isinstance(dashed, dict):
            dash_map = {lvl: dashed.get(lvl, "") for lvl in present_levels}
        else:
            if isinstance(dashed, str):
                dashed = [dashed]
            dashed = set(dashed)
            dash_map = {lvl: (dashed_pattern if lvl in dashed else "") for lvl in present_levels}

    # keep text editable in SVG (nice for Illustrator)
    if export_format == "svg" and keep_text_as_text:
        import matplotlib as mpl
        mpl.rcParams["svg.fonttype"] = "none"

    # --- NEW: build a seaborn palette aligned to present_levels -------------
    if isinstance(fallback_palette, str):
        fb_list = sns.color_palette(fallback_palette, n_colors=max(1, len(present_levels)))
    else:
        # e.g., a list of colors provided manually
        fb_list = list(fallback_palette)

    palette_to_use = {}
    for i, lvl in enumerate(present_levels):
        if color_map is not None and lvl in color_map and color_map[lvl]:
            palette_to_use[lvl] = color_map[lvl]
        else:
            palette_to_use[lvl] = fb_list[i % len(fb_list)]

    # plot --------------------------------------------------------------------
    plt.figure(figsize=(12, 8))
    sns.lineplot(
        data=df,
        x="TrainSize",
        y="metric_value",
        hue="pretraining",
        hue_order=present_levels,
        palette=palette_to_use,                      # <- NEW: manual colors
        style="pretraining" if dash_map is not None else None,
        style_order=present_levels if dash_map is not None else None,
        dashes=dash_map if dash_map is not None else False,
        err_style="bars",
        err_kws={"capsize": 3},
        errorbar=("sd"),
    )

    # labels & axes
    plt.xlabel(xlabel, fontsize=20)
    plt.ylabel(ylabel, fontsize=20)
    plt.title("")
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)

    # LEGEND: fully adjustable
    if legend_bbox is not None:
        plt.legend(title="", fontsize=legend_fontsize, ncol=legend_ncol,
                   loc=legend_loc, bbox_to_anchor=legend_bbox, borderaxespad=0.0)
    else:
        plt.legend(title="", fontsize=legend_fontsize, ncol=legend_ncol, loc=legend_loc)

    plt.ylim(0, 1)
    plt.grid(False)
    plt.tight_layout()

    # EXPORT
    if export_path is not None:
        plt.savefig(
            export_path,
            format=export_format,
            dpi=export_dpi,
            bbox_inches="tight",
            transparent=export_transparent,
        )

    plt.show()
    return df

# Call the function with the example parameters
df_result = plot_finetuning_results_slides(
    folder_path, xlabel, ylabel, title_notes, title,
    None, xgb_res_path, lo_bound, hi_bound, custom_order
)
